# 🌾 Andhra Pradesh Paddy 3-Day Price Prediction Engine
### Data.gov.in API Integration | Global XGBoost | Quantile Risk Bands

This notebook fetches live data from the Data.gov.in API using the official endpoint, parses the schema correctly, filters it to recent price regimes (2018+), and trains a single Global XGBoost model to generate recursive 3-day forecasts with Worst-Case, Expected, and Best-Case prices for farmers.

## Step 1: Fetch Data from Data.gov.in API

Uses the official Data.gov.in parameters to fetch all `Andhra Pradesh` / `Paddy(Common)` records. Paginates through the results until all records are collected.

In [ ]:
import urllib.request
import urllib.parse
import json
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

API_KEY = "579b464db66ec23bdd000001a0a99e04a75a40666201931688acb738"
RESOURCE_ID = "35985678-0d79-46b4-9ed6-6f13308a1d24"
BASE_URL = f"https://api.data.gov.in/resource/{RESOURCE_ID}"

def fetch_mandi_data(state="Andhra Pradesh", commodity="Paddy(Common)"):
    all_records = []
    offset = 0
    limit = 1000
    total = None
    
    print(f"Fetching data for {state} - {commodity}...")
    
    while True:
        params = {
            "api-key": API_KEY,
            "format": "json",
            "limit": limit,
            "offset": offset,
            "filters[state]": state,
            "filters[commodity]": commodity
        }
        url = f"{BASE_URL}?{urllib.parse.urlencode(params)}"
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        
        try:
            with urllib.request.urlopen(req, timeout=15) as response:
                data = json.loads(response.read().decode('utf-8'))
                records = data.get("records", [])
                
                if total is None:
                    total = int(data.get("total", 0))
                    print(f"Total available API records: {total}")
                    
                if not records:
                    break
                
                # CRITICAL FIX: The data.gov.in API has a known bug where
                # filters[state] stops working past the total-count offset,
                # returning records from other states (MP, UP, etc.).
                # Filter each page to only keep records matching our state.
                valid_records = [r for r in records if r.get('State', '') == state]
                
                # If NONE of the records on this page match our state,
                # the API has overflowed - stop pagination immediately.
                if len(valid_records) == 0:
                    print(f"[STOP] API overflow at offset {offset} - records belong to other states. Stopping.")
                    break
                
                # Warn if some records on this page were from wrong states
                dropped = len(records) - len(valid_records)
                if dropped > 0:
                    print(f"  [FILTER] Dropped {dropped}/{len(records)} non-{state} records at offset {offset}")
                    
                all_records.extend(valid_records)
                print(f"Fetched {len(all_records)} / {total} records...")
                
                offset += limit
                if offset >= total:
                    break
                    
                time.sleep(0.2)
        except Exception as e:
            print(f"Error fetching offset {offset}: {e}")
            time.sleep(1.0)
            # Retry once
            try:
                with urllib.request.urlopen(req, timeout=15) as response:
                    data = json.loads(response.read().decode('utf-8'))
                    records = data.get("records", [])
                    if records:
                        valid_records = [r for r in records if r.get('State', '') == state]
                        all_records.extend(valid_records)
                        offset += limit
            except Exception as e2:
                print(f"Retry failed: {e2}")
                break
                
    df = pd.DataFrame(all_records)
    
    # SAFETY NET: Final hard filter to guarantee only our target state remains
    if 'State' in df.columns and not df.empty:
        before = len(df)
        df = df[df['State'] == state].reset_index(drop=True)
        after = len(df)
        if before != after:
            print(f"[SAFETY] Post-fetch filter removed {before - after} non-{state} rows.")
    
    return df

# Execute fetch
df = fetch_mandi_data(state="Andhra Pradesh", commodity="Paddy(Common)")
print(f"\n[OK] Total records fetched: {len(df)}")
if 'State' in df.columns:
    print(f"States in dataset: {df['State'].unique().tolist()}")
    print(f"Markets found: {sorted(df['Market'].unique().tolist())}")

## Step 2: Data Cleaning & Regime Filtering

Maps the exact API schema (`Arrival_Date`, `Market`, `Modal_Price`). Parses `DD/MM/YYYY` dates correctly, cleans market names, and filters to 2018-Present to match current MSP regimes.

In [ ]:
if df.empty:
    raise ValueError("DataFrame is empty. API fetch failed.")

# 1. Parse Date (API format is DD/MM/YYYY)
df['date'] = pd.to_datetime(df['Arrival_Date'], format='%d/%m/%Y', errors='coerce')

# 2. Standardize Market Name (Remove ' APMC' suffix for cleaner columns later)
df['Market'] = df['Market'].astype(str).str.replace(' APMC', '').str.strip()

# 3. Convert Prices to Numeric
df['weighted_avg_modal_price'] = pd.to_numeric(df['Modal_Price'], errors='coerce')
df['min_price'] = pd.to_numeric(df['Min_Price'], errors='coerce')
df['max_price'] = pd.to_numeric(df['Max_Price'], errors='coerce')

# 4. HARD STATE FILTER - Guarantee only Andhra Pradesh data
if 'State' in df.columns:
    before_state_filter = len(df)
    df = df[df['State'] == 'Andhra Pradesh']
    dropped_states = before_state_filter - len(df)
    if dropped_states > 0:
        print(f"\u26a0\ufe0f Removed {dropped_states} rows from non-AP states (API overflow bug)")
    print(f"\u2705 Verified: All {len(df)} rows belong to Andhra Pradesh")

# Drop rows with missing modal price or date
df = df.dropna(subset=['date', 'weighted_avg_modal_price'])

# Filter to 2018+ to match current MSP/price regimes
df = df[df['date'] >= '2018-01-01'].sort_values(['Market', 'date']).reset_index(drop=True)

print(f"[OK] Filtered to {len(df)} rows from {df['Market'].nunique()} AP markets (2018-Present).")
print("Markets found:", sorted(df['Market'].unique().tolist()))

## Step 3: Feature Engineering (No Arrivals to Avoid Leakage)

Creates price lags, momentum (rolling means), and calendar features. One-hot encodes the `Market` column so a single global model can handle all mandis without maintaining separate models.

In [ ]:
def create_features(df):
    df = df.sort_values(['Market', 'date']).copy()
    
    # Price Lags
    for lag in [1, 2, 3, 7]:
        df[f'lag_{lag}'] = df.groupby('Market')['weighted_avg_modal_price'].shift(lag)
        
    # Rolling Means (Momentum)
    df['rolling_mean_3'] = df.groupby('Market')['lag_1'].transform(lambda x: x.rolling(3).mean())
    df['rolling_mean_7'] = df.groupby('Market')['lag_1'].transform(lambda x: x.rolling(7).mean())
    
    # Calendar Features
    df['dayofweek'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    
    return df.dropna()

featured_df = create_features(df)

# --- CRITICAL: Use One-Hot Encoding instead of enable_categorical ---
featured_df = pd.get_dummies(featured_df, columns=['Market'], prefix='mkt', drop_first=False)
print("\u2705 Features created and markets one-hot encoded.")

## Step 4: Train Global XGBoost & Evaluate vs Baseline

Trains the quantile models (with automatic fallback to squared error if Colab's XGBoost version requires it). Compares the ML model against a Naive Baseline (Yesterday's Price).

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error

# Define Features (Exclude raw API columns and targets)
exclude_cols = ['date', 'weighted_avg_modal_price', 'min_price', 'max_price', 
                'District', 'State', 'Commodity', 'Variety', 'Grade',
                'Arrival_Date', 'Market'] # 'Market' is excluded because it's one-hot encoded
                
FEATURES = [c for c in featured_df.columns if c not in exclude_cols and featured_df[c].dtype != 'object']

# Chronological 80/20 Split
split_date = featured_df['date'].quantile(0.8)
train_df = featured_df[featured_df['date'] <= split_date]
test_df = featured_df[featured_df['date'] > split_date]

X_train, y_train = train_df[FEATURES], train_df['weighted_avg_modal_price']
X_test, y_test = test_df[FEATURES], test_df['weighted_avg_modal_price']

# Train Models (with Fallback Logic)
params = {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 200, 'random_state': 42}
models = {}
val_mae = 0
use_quantile = False

try:
    # Try modern Quantile API
    print("Attempting Quantile Regression...")
    models['p50'] = xgb.XGBRegressor(objective='reg:quantile', quantile_alpha=0.5, **params)
    models['p10'] = xgb.XGBRegressor(objective='reg:quantile', quantile_alpha=0.1, **params)
    models['p90'] = xgb.XGBRegressor(objective='reg:quantile', quantile_alpha=0.9, **params)
    
    models['p50'].fit(X_train, y_train)
    models['p10'].fit(X_train, y_train)
    models['p90'].fit(X_train, y_train)
    use_quantile = True
    print("[OK] Quantile models trained successfully.")
    
except Exception as e:
    # Fallback to standard Squared Error
    print(f"[WARNING] Quantile failed ({e}). Falling back to squared error + MAE bounds.")
    models['p50'] = xgb.XGBRegressor(objective='reg:squarederror', **params)
    models['p50'].fit(X_train, y_train)
    
    # Calculate MAE on validation to create artificial bounds
    val_preds = models['p50'].predict(X_train)
    val_mae = np.mean(np.abs(y_train - val_preds))

# Evaluation vs Baseline
pred_50 = models['p50'].predict(X_test)
baseline_pred = X_test['lag_1']

ml_mape = mean_absolute_percentage_error(y_test, pred_50) * 100
base_mape = mean_absolute_percentage_error(y_test, baseline_pred) * 100

print(f"\n--- Evaluation ---")
print(f"Baseline (Naive Lag-1) MAPE: {base_mape:.2f}%")
print(f"Global XGBoost (Median) MAPE: {ml_mape:.2f}%")
if ml_mape < base_mape:
    print("[OK] ML Model beats the baseline!")
else:
    print("[WARNING] Baseline beats ML. Consider using Naive Lag-1 as the primary forecast.")

## Step 5: The 3-Day Farmer Forecast Engine

Generates recursive 3-day forecasts for an individual market, providing Expected, Worst-Case, and Best-Case prices to help farmers manage risk.

In [ ]:
def predict_farmer_3day(market_name):
    # Find the one-hot encoded column
    mkt_col = f'mkt_{market_name}'
    if mkt_col not in featured_df.columns:
        return f"Market '{market_name}' not found in dataset."
        
    mkt_hist = featured_df[featured_df[mkt_col] == 1].sort_values('date')
    
    if mkt_hist.empty:
        return f"No historical data for {market_name}."
        
    latest_row = mkt_hist.iloc[-1].copy()
    predictions = []
    
    for day in range(1, 4):
        X_live = pd.DataFrame([latest_row])[FEATURES]
        
        p50 = float(models['p50'].predict(X_live)[0])
        
        if use_quantile:
            p10 = float(models['p10'].predict(X_live)[0])
            p90 = float(models['p90'].predict(X_live)[0])
        else:
            # Fallback bounds using validation MAE
            p10 = p50 - (1.5 * val_mae)
            p90 = p50 + (1.5 * val_mae)
            
        p10 = min(p10, p50)
        p90 = max(p90, p50)
        
        predictions.append({
            'Day': f"Day +{day}",
            'Expected_Price (Rs/Quintal)': round(p50, 2),
            'Worst_Case (Rs/Quintal)': round(p10, 2),
            'Best_Case (Rs/Quintal)': round(p90, 2),
            'Risk_Advice': "High risk of price drop" if (p50 - p10) > 50 else "Stable expected price"
        })
        
        # Recursive Update (Approximate for 3 days)
        latest_row['lag_7'] = latest_row.get('lag_6', latest_row['lag_3'])
        latest_row['lag_3'] = latest_row['lag_2']
        latest_row['lag_2'] = latest_row['lag_1']
        latest_row['lag_1'] = p50
        latest_row['rolling_mean_3'] = (latest_row['lag_1'] + latest_row['lag_2'] + latest_row['lag_3']) / 3.0
        latest_row['rolling_mean_7'] = (latest_row['lag_1'] * 3 + latest_row['rolling_mean_3'] * 4) / 7.0
        
    return pd.DataFrame(predictions)

# --- Run Forecasts for ALL Individual Markets ---
all_markets = sorted([c.replace('mkt_', '') for c in FEATURES if c.startswith('mkt_')])

print("="*70)
print("GLOBAL MODEL INDIVIDUAL MARKET 3-DAY FORECASTS")
print("="*70)

for market in all_markets:
    print(f"\n=== 3-Day Forecast for {market} ===")
    forecast_df = predict_farmer_3day(market)
    if isinstance(forecast_df, pd.DataFrame):
        print(forecast_df.to_string(index=False))
    else:
        print(forecast_df)
    print("-" * 70)